# IPSD — Minggu 2: Pengolahan/Praproses Data

**Mata Kuliah:** Infrastruktur dan Platform untuk Sains Data  
**Sub-CLO:** Sub-CLO-09-2-1  
**AI Level:** L2  

**AI Level 2 — AI-Assisted Idea Generation and Structuring.** AI boleh digunakan untuk brainstorming, decomposition, alternatif struktur/pipeline, dan pertanyaan klarifikasi. Jawaban akhir, kode akhir, keputusan, dan interpretasi harus dibuat mahasiswa. Simpan prompt log.

## Capaian Praktikum
- Mengidentifikasi masalah kualitas data.
- Melakukan penanganan missing value, duplicate, tipe data, encoding, scaling, dan outlier sederhana.
- Menyusun preprocessing pipeline yang dapat dijelaskan.

**Dataset:** Breast Cancer Wisconsin dari scikit-learn, dimodifikasi dengan missing value dan duplikasi sintetis.

## Alur Praktikum
- 1. Load dataset built-in scikit-learn.
- 2. Sisipkan missing value dan duplikasi untuk latihan.
- 3. Audit kualitas data dengan `info()`, `isna()`, `duplicated()`, dan statistik deskriptif.
- 4. Lakukan imputasi, hapus duplikasi, dan scaling pada fitur numerik.

In [8]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer(as_frame=True)
df = data.frame.copy()
# Sisipkan masalah kualitas data untuk latihan
rng = np.random.default_rng(42)
idx = rng.choice(df.index, size=15, replace=False)
df.loc[idx, 'mean radius'] = np.nan
df = pd.concat([df, df.iloc[:3]], ignore_index=True)
print(df.shape)
print('Missing:', int(df.isna().sum().sum()))
print('Duplicate:', int(df.duplicated().sum()))

# Contoh cleaning dasar
df = df.drop_duplicates().reset_index(drop=True)
imputer = SimpleImputer(strategy='median')
df[['mean radius']] = imputer.fit_transform(df[['mean radius']])
print('Sesudah cleaning:', df.shape, int(df.isna().sum().sum()))

df

(572, 31)
Missing: 15
Duplicate: 3
Sesudah cleaning: (569, 31) 0


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,0.1726,0.05623,...,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115,0
565,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,0.1752,0.05533,...,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637,0
566,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,0.1590,0.05648,...,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820,0
567,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,0.2397,0.07016,...,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400,0


STEP 1 : Load dataset built-in scikit-learn

In [31]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer(as_frame=True)
df = data.frame.copy()
print('Shape awal:', df.shape)
df.head()


Shape awal: (569, 31)


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


STEP 2 : Sisipkan missing value dan duplikasi untuk latihan

In [32]:
# Sisipkan missing value di beberapa kolom berbeda
df.loc[[8, 19, 77, 165, 240], 'mean radius'] = np.nan       # 5 baris pada kolom mean radius
df.loc[[12, 45, 130, 310, 450], 'mean texture'] = np.nan    # 5 baris pada kolom mean texture
df.loc[[30, 95, 190, 275, 480], 'mean area'] = np.nan       # 5 baris pada kolom mean area

# Sisipkan duplikasi lewat penugasan baris langsung
df.loc[9] = df.loc[10]      # duplikat baris 10 ke baris 9
df.loc[40] = df.loc[41]     # duplikat baris 41 ke baris 40
df.loc[110] = df.loc[200]   # duplikat baris 200 ke baris 110
df.loc[260] = df.loc[15]    # duplikat baris 15 ke baris 260
df.loc[390] = df.loc[391]   # duplikat baris 391 ke baris 390
df.loc[520] = df.loc[521]   # duplikat baris 521 ke baris 520

print('Total missing value:', int(df.isna().sum().sum()))
print('Total baris duplikat:', int(df.duplicated().sum()))
df.head(10)


Total missing value: 15
Total baris duplikat: 6


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.26540,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.18600,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.24300,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.25750,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.16250,0.2364,0.07678,0
5,12.45,15.70,82.57,477.1,0.12780,0.17000,0.15780,0.08089,0.2087,0.07613,...,23.75,103.40,741.6,0.1791,0.5249,0.5355,0.17410,0.3985,0.12440,0
6,18.25,19.98,119.60,1040.0,0.09463,0.10900,0.11270,0.07400,0.1794,0.05742,...,27.66,153.20,1606.0,0.1442,0.2576,0.3784,0.19320,0.3063,0.08368,0
7,13.71,20.83,90.20,577.9,0.11890,0.16450,0.09366,0.05985,0.2196,0.07451,...,28.14,110.60,897.0,0.1654,0.3682,0.2678,0.15560,0.3196,0.11510,0
8,NaN,21.82,87.50,519.8,0.12730,0.19320,0.18590,0.09353,0.2350,0.07389,...,30.73,106.20,739.3,0.1703,0.5401,0.5390,0.20600,0.4378,0.10720,0
9,16.02,23.24,102.70,797.8,0.08206,0.06669,0.03299,0.03323,0.1528,0.05697,...,33.88,123.80,1150.0,0.1181,0.1551,0.1459,0.09975,0.2948,0.08452,0


STEP 3 : Audit kualitas data dengan `info()`, `isna()`, `duplicated()`, dan statistik deskriptif

In [33]:
# a) Struktur & tipe data
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   mean radius              564 non-null    float64
 1   mean texture             564 non-null    float64
 2   mean perimeter           569 non-null    float64
 3   mean area                564 non-null    float64
 4   mean smoothness          569 non-null    float64
 5   mean compactness         569 non-null    float64
 6   mean concavity           569 non-null    float64
 7   mean concave points      569 non-null    float64
 8   mean symmetry            569 non-null    float64
 9   mean fractal dimension   569 non-null    float64
 10  radius error             569 non-null    float64
 11  texture error            569 non-null    float64
 12  perimeter error          569 non-null    float64
 13  area error               569 non-null    float64
 14  smoothness error         5

In [34]:
# b) Cek missing value per kolom (yang > 0 saja)
missing_per_kolom = df.isna().sum()
print('Kolom dengan missing value:')
print(missing_per_kolom[missing_per_kolom > 0])
print('\nTotal missing value:', int(df.isna().sum().sum()))



Kolom dengan missing value:
mean radius     5
mean texture    5
mean area       5
dtype: int64

Total missing value: 15


In [35]:
# c) Cek baris duplikat
print('Jumlah baris duplikat:', int(df.duplicated().sum()))
df[df.duplicated(keep=False)].sort_values(by=list(df.columns[:3]))



Jumlah baris duplikat: 6


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
390,8.734,16.84,55.27,234.3,0.10390,0.07428,0.00000,0.00000,0.1985,0.07098,...,22.80,64.01,317.0,0.1460,0.1310,0.0000,0.00000,0.2445,0.08865,1
391,8.734,16.84,55.27,234.3,0.10390,0.07428,0.00000,0.00000,0.1985,0.07098,...,22.80,64.01,317.0,0.1460,0.1310,0.0000,0.00000,0.2445,0.08865,1
40,10.950,21.35,71.90,371.1,0.12270,0.12180,0.10440,0.05669,0.1895,0.06870,...,35.34,87.22,514.0,0.1909,0.2698,0.4023,0.14240,0.2964,0.09606,0
41,10.950,21.35,71.90,371.1,0.12270,0.12180,0.10440,0.05669,0.1895,0.06870,...,35.34,87.22,514.0,0.1909,0.2698,0.4023,0.14240,0.2964,0.09606,0
110,12.230,19.56,78.54,461.0,0.09586,0.08087,0.04187,0.04107,0.1979,0.06013,...,28.36,92.15,638.4,0.1429,0.2042,0.1377,0.10800,0.2668,0.08174,1
200,12.230,19.56,78.54,461.0,0.09586,0.08087,0.04187,0.04107,0.1979,0.06013,...,28.36,92.15,638.4,0.1429,0.2042,0.1377,0.10800,0.2668,0.08174,1
15,14.540,27.54,96.73,658.8,0.11390,0.15950,0.16390,0.07364,0.2303,0.07077,...,37.13,124.10,943.2,0.1678,0.6577,0.7026,0.17120,0.4218,0.13410,0
260,14.540,27.54,96.73,658.8,0.11390,0.15950,0.16390,0.07364,0.2303,0.07077,...,37.13,124.10,943.2,0.1678,0.6577,0.7026,0.17120,0.4218,0.13410,0
9,16.020,23.24,102.70,797.8,0.08206,0.06669,0.03299,0.03323,0.1528,0.05697,...,33.88,123.80,1150.0,0.1181,0.1551,0.1459,0.09975,0.2948,0.08452,0
10,16.020,23.24,102.70,797.8,0.08206,0.06669,0.03299,0.03323,0.1528,0.05697,...,33.88,123.80,1150.0,0.1181,0.1551,0.1459,0.09975,0.2948,0.08452,0


In [36]:
# d) Statistik deskriptif fitur numerik
df.describe().T

,count,mean,std,min,25%,50%,75%,max
mean radius,564.0,14.143246,3.549694,6.981000,11.697500,13.355000,15.892500,28.11000
mean texture,564.0,19.315266,4.289589,9.710000,16.202500,18.880000,21.802500,39.28000
mean perimeter,569.0,92.108541,24.411394,43.790000,75.210000,86.490000,104.100000,188.50000
mean area,564.0,655.933688,354.050973,143.500000,420.175000,551.400000,784.150000,2501.00000
mean smoothness,569.0,0.096326,0.013992,0.052630,0.086370,0.095860,0.105300,0.16340
mean compactness,569.0,0.104382,0.052741,0.019380,0.065260,0.092630,0.130500,0.34540
mean concavity,569.0,0.088919,0.079751,0.000000,0.029560,0.061550,0.130700,0.42680
mean concave points,569.0,0.049079,0.038949,0.000000,0.020370,0.033840,0.073640,0.20120
mean symmetry,569.0,0.181245,0.027469,0.106000,0.161900,0.179300,0.195900,0.30400
mean fractal dimension,569.0,0.062776,0.006994,0.049960,0.057800,0.061540,0.066150,0.09744


STEP 4 : Lakukan imputasi, hapus duplikasi, dan scaling pada fitur numerik

In [18]:
# a) Hapus baris duplikat
df = df.drop_duplicates().reset_index(drop=True)
print('Shape setelah drop_duplicates:', df.shape)
print('Duplicate tersisa:', int(df.duplicated().sum()))

# b) Imputasi missing value pada kolom-kolom yang berlubang (strategi median)
kolom_missing = ['mean radius', 'mean texture', 'mean area']
imputer = SimpleImputer(strategy='median')
df[kolom_missing] = imputer.fit_transform(df[kolom_missing])

print('Sesudah imputasi -> shape:', df.shape, '| Missing tersisa:', int(df.isna().sum().sum()))

# c) Scaling pada fitur numerik (kecuali kolom target)
fitur_numerik = [c for c in df.columns if c != 'target']

scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[fitur_numerik] = scaler.fit_transform(df[fitur_numerik])

print('Contoh sebelum scaling:')
print(df[fitur_numerik].describe().T[['mean', 'std', 'min', 'max']].head())

print('\nContoh sesudah scaling (mean~0, std~1):')
print(df_scaled[fitur_numerik].describe().T[['mean', 'std', 'min', 'max']].head())




Shape setelah drop_duplicates: (563, 31)
Duplicate tersisa: 0
Sesudah imputasi -> shape: (563, 31) | Missing tersisa: 0
Contoh sebelum scaling:
                       mean         std        min        max
mean radius       14.110876    3.474293    6.98100    28.1100
mean texture      19.285710    4.273061    9.71000    39.2800
mean perimeter    92.076590   24.269044   43.79000   188.5000
mean area        654.244405  350.152068  143.50000  2501.0000
mean smoothness    0.096249    0.013983    0.05263     0.1634

Contoh sesudah scaling (mean~0, std~1):
                         mean       std       min       max
mean radius     -3.912402e-16  1.000889 -2.054005  4.032928
mean texture    -4.732745e-18  1.000889 -2.242941  4.683310
mean perimeter  -4.291022e-16  1.000889 -1.991406  3.976636
mean area        1.956201e-16  1.000889 -1.459933  5.278843
mean smoothness -1.893098e-16  1.000889 -3.122086  4.806473


## Latihan Terstruktur
- Bandingkan minimal dua strategi imputasi pada satu fitur.
- Tentukan kapan scaling diperlukan dan kapan tidak.
- Buat tabel before/after berisi jumlah missing, duplicate, dan rentang nilai fitur.

In [46]:
# Latihan 1: Bandingkan minimal dua strategi imputasi pada satu fitur ('mean radius')
data_raw = load_breast_cancer(as_frame=True)
df_raw = data_raw.frame.copy()

df_raw.loc[[8, 19, 77, 165, 240], 'mean radius'] = np.nan
df_raw.loc[[12, 45, 130, 310, 450], 'mean texture'] = np.nan
df_raw.loc[[30, 95, 190, 275, 480], 'mean area'] = np.nan

df_raw.loc[9] = df_raw.loc[10]
df_raw.loc[40] = df_raw.loc[41]
df_raw.loc[110] = df_raw.loc[200]
df_raw.loc[260] = df_raw.loc[15]
df_raw.loc[390] = df_raw.loc[391]
df_raw.loc[520] = df_raw.loc[521]

# Catat kondisi SEBELUM cleaning (fokus ke 'mean radius')
before_missing = int(df_raw['mean radius'].isna().sum())
before_duplicate = int(df_raw.duplicated().sum())
before_min = df_raw['mean radius'].min()
before_max = df_raw['mean radius'].max()

print('Sebelum cleaning')
print('Missing (mean radius):', before_missing)
print('Duplicate rows:', before_duplicate)
print('Rentang mean radius:', before_min, '-', before_max)


Sebelum cleaning
Missing (mean radius): 5
Duplicate rows: 6
Rentang mean radius: 6.981 - 28.11


In [41]:
# Strategi 1: Imputasi MEDIAN
df_median = df_raw.drop_duplicates().reset_index(drop=True).copy()
imputer_median = SimpleImputer(strategy='median')
df_median[['mean radius']] = imputer_median.fit_transform(df_median[['mean radius']])

# Strategi 2: Imputasi MEAN
df_mean = df_raw.drop_duplicates().reset_index(drop=True).copy()
imputer_mean = SimpleImputer(strategy='mean')
df_mean[['mean radius']] = imputer_mean.fit_transform(df_mean[['mean radius']])

print('Nilai pengganti (median):', round(imputer_median.statistics_[0], 4))
print('Nilai pengganti (mean)  :', round(imputer_mean.statistics_[0], 4))
print('Selisih median vs mean  :', round(abs(imputer_median.statistics_[0] - imputer_mean.statistics_[0]), 4))


Nilai pengganti (median): 13.355
Nilai pengganti (mean)  : 14.1392
Selisih median vs mean  : 0.7842


**ANALISIS**

1. imputasi (median vs mean):

nilai median 13,36 sama mean 14,14 itu beda 0,78, jadi kelihatan ada outlier di data mean radius, makanya median lebih aman dipake soalnya median gak gampang keganggu sama nilai ekstrem 

2. scaling:

kalo diliat rentang fitur, mean area itu bisa sampe ratusan bahkan ribuan, sedangkan mean smoothness cuma di bawah 1. makanya model yg ngitung pake jarak kayak KNN atau SVM wajib di scaling biar semua fiturny setara

3. tabel before/after:

sebelum cleaning ada 5 missing value di mean radius sama 6 baris duplikat, total datanya 569 baris. abis di drop_duplicates sm diimputasi, missingnya jadi 0, duplikat juga 0, rentang mean radiusnya sma aja , cuma jumlah barisnya turun jadi 563 karena yg duplikat dibuang

untuk pipeline final aku milih median karena data mean radius keliatan ad aoutlier dri selisih diatas, jadi median lebih cocok buat missing valueny

In [1]:
# Latihan 2: Tentukan kapan scaling diperlukan dan kapan tidak

df_clean = df_median.copy()

fitur_numerik = ['mean radius', 'mean area', 'mean smoothness']
print('Sebelum scaling:')
print(df_clean[fitur_numerik].describe().loc[['min', 'max', 'mean', 'std']])

scaler = StandardScaler()
df_scaled = df_clean.copy()
df_scaled[fitur_numerik] = scaler.fit_transform(df_clean[fitur_numerik])

print('\nSesudah scaling:')
print(df_scaled[fitur_numerik].describe().loc[['min', 'max', 'mean', 'std']])



NameError: name 'df_median' is not defined

**Kapan scaling diperlukan vs tidak (isi/adaptasi alasanmu):**
- **Perlu scaling**: model berbasis jarak/gradien seperti KNN, SVM, Logistic Regression (dengan regularisasi), PCA, dan neural network — karena fitur dengan skala besar (mis. `mean area` bisa ratusan-ribuan) akan mendominasi perhitungan jarak/gradien dibanding fitur berskala kecil (mis. `mean smoothness` yang di bawah 1).
- **Tidak wajib scaling**: model berbasis pohon seperti Decision Tree, Random Forest, Gradient Boosting — karena split dilakukan berdasarkan urutan nilai per fitur, bukan magnitudonya.
- Contoh di atas: perhatikan `mean area` punya rentang jauh lebih besar dari `mean smoothness` sebelum scaling; setelah `StandardScaler`, keduanya berada di skala yang sebanding (mean≈0, std≈1).

In [43]:
# Latihan 3: Tabel before/after (missing, duplicate, rentang nilai fitur)

after_missing = int(df_clean['mean radius'].isna().sum())
after_duplicate = int(df_clean.duplicated().sum())
after_min = df_clean['mean radius'].min()
after_max = df_clean['mean radius'].max()

summary_table = pd.DataFrame({
    'Tahap': ['Sebelum cleaning', 'Sesudah cleaning (median imputation + drop_duplicates)'],
    'Missing (mean radius)': [before_missing, after_missing],
    'Duplicate rows': [before_duplicate, after_duplicate],
    'Min mean radius': [round(before_min, 4), round(after_min, 4)],
    'Max mean radius': [round(before_max, 4), round(after_max, 4)],
    'Jumlah baris': [len(df_raw), len(df_clean)],
})
summary_table


,Tahap,Missing (mean radius),Duplicate rows,Min mean radius,Max mean radius,Jumlah baris
0,Sebelum cleaning,5,6,6.981,28.11,569
1,Sesudah cleaning (median imputation + drop_dup...,0,0,6.981,28.11,563


## Luaran
Notebook + tabel before/after + prompt log L2.

**Bukti AI/validasi:** Prompt log; alasan memilih preprocessing.


## Prompt Log — wajib

| No | Tujuan Prompt | Prompt | Ringkasan Respons AI | Keputusan Saya / Apa yang Dipakai | Verifikasi |
|---|---|---|---|---|---|
| 1 | Memahami alur dan instruksi tugas praktikum | "ini aku disuruh ngerjain apa ya? tolong dong ini aku dijelasin alur nya gimna.. gapaham" | AI menjelaskan alur praktikum secara berurutan mulai dari running contoh dasar, pengerjaan latihan terstruktur (imputasi, scaling, tabel before/after), hingga pengisian prompt log dan refleksi. | Saya mengikuti alur kerja yang dijelaskan dan mulai mengerjakan latihan terstruktur di sel `# TODO`. | Mengecek kesesuaian penjelasan AI dengan instruksi pada notebook IPSD Minggu 2. |
| 2 | Memahami cara membandingkan 2 strategi imputasi | "okay, yg step 1 uda selesai dan done. skrg yg step 2 yg pertama.. badingin 2 strategi imputasi.. itu caranya gimana" | AI menjelaskan konsep perbandingan strategi imputasi (`mean` vs `median`) dan memberikan contoh kode Python untuk membandingkan statistik hasilnya. | Saya menggunakan pendekatan perbandingan `mean` dan `median` pada fitur `mean radius`. | Memastikan kode dapat dieksekusi tanpa error di Jupyter Notebook dan memberikan output statistik. |
| 3 | Mengonfirmasi keberadaan instruksi di kode bawaan | "dari step yg bandingin minim 2 data kok gaada ini ya" / "ya sama aja, harus ada itu.. itu bukan prnjrlasan umum nya tpi di code nya juga" | AI menjelaskan bahwa kode bawaan dosen hanya contoh dasar (1 strategi), sedangkan perbandingan 2 strategi memang bagian latihan yang harus ditulis sendiri oleh mahasiswa. | Saya menyatukan seluruh kode perbandingan imputasi, cleaning, scaling, dan tabel before/after ke dalam sel `# TODO`. | Menjalankan seluruh sel kode `# TODO` dan memastikan output tabel perbandingan serta tabel before/after muncul dengan benar. |


## Refleksi Singkat

1. **Bagian apa yang paling terbantu oleh AI?**  
   Bagian pemahaman alur kerja praktikum, menanyakan bagaimana membuat struktur kode Python tentang perbandingan imputasi, scaling, tabel before/after, dan alurnya

2. **Bagian apa yang tetap harus Anda putuskan sendiri?**  
   Pemilihan strategi imputasi akhir yang paling sesuai untuk dataset (memilih `median` karena tahan terhadap *outlier*) serta memastikan logika kode berjalan sesuai dengan yang diinstruksikan 

3. **Apa satu keterbatasan/risiko respons AI yang Anda temukan?**  
   AI awalnya berasumsi beberapa kode latihan sudah ada di dalam template awal, sehingga saya harus menglarifikasi dan memastikan sendiri kode mana yang perlu ditambahkan di sel `# TODO`.